# Theoretical Foundations of Manifold Learning

**Angela Lavinia Varazi** · Bachelor of Science thesis · Bocconi University · Academic year 2024–2025

## About this notebook

The nine code cells below reproduce the code printed in Appendix A of `ManifoldLearning.pdf` (physical PDF pages 56–62), in its original order. Code, comments, spelling, parameters, and cell boundaries are preserved; indentation and visible blank lines are recovered from the PDF layout. No algorithmic changes or extra executable setup cells have been added.

This is a reconstruction from the printed PDF, not the original notebook file. Execution counts and outputs are cleared. The original plots remain in the thesis. Added introductory notes and cell metadata describe the reconstruction.

## Before running

Install the packages listed in `requirements.txt`. Download the [COIL-100 dataset from Columbia University](https://www.cs.columbia.edu/CAVE/software/softlib/coil-100.php) and place the PNG images in `coil-100/coil-100/` relative to this notebook, matching the original code. For example: `coil-100/coil-100/obj5__0.png`.

Run the cells in order. The original setting is `object_label = 5`. The later methods reuse variables defined in earlier cells.

The notebook structure and Python syntax have been checked. The code has not been executed against COIL-100, and the original package versions are not recorded in the PDF.


In [ ]:
object_label = 5

## K-nearest neighbors


In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.neighbors import kneighbors_graph
from scipy.sparse.linalg import eigsh
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.colors as mcolors


 
coil_100_dir = './coil-100/coil-100'


images = []
angles = []  #  rotation angles
 
#object_label = 10  # object number in the coil dataset
for degree in range(0, 360, 5):  #  0, 5, 10, ..., 355 degrees
    image_filename = f'obj{object_label}__{degree}.png'
    image_path = os.path.join(coil_100_dir, image_filename)
    
    if os.path.exists(image_path):
        img = Image.open(image_path).resize((128, 128))  # resolution (128x128)
        img_array = np.array(img).flatten()
        images.append(img_array)
        angles.append(degree)  # store the rotation angle as label

 
X_subset = np.array(images)  
angles = np.array(angles)  

print(images[0])

 
k = 10 #   neighbors
W = kneighbors_graph(X_subset, k, mode='connectivity', include_self=True).toarray()

 
D = np.diag(W.sum(axis=1))   
L = D - W  # unnormalized Laplacian

 
n_components = 3  #  
eigvals, eigvecs = eigsh(L, k=n_components+1, which='SM')  #  smallest eigenvalues
embedding = eigvecs[:, 1:]  # skiping  the first eigenvector (it's const func)

 
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Normalize colors to span full range (0 to 360 degrees)
norm = mcolors.Normalize(vmin=0, vmax=360)
cmap = plt.get_cmap('jet')  # 'jet' provides full bright spectrum

sc = ax.scatter(embedding[:, 0], embedding[:, 1], embedding[:, 2], 
                c=angles, cmap=cmap, norm=norm, edgecolor='k', s=50)

 
cbar = plt.colorbar(sc, ax=ax, shrink=0.6)
cbar.set_label('Rotation Angle (Degrees)')

ax.set_title('Laplacian Eigenmaps Projection (3D) - Object ' + str(object_label) +' (72 Images)')
ax.set_xlabel('Eigenvector 1')
ax.set_ylabel('Eigenvector 2')
ax.set_zlabel('Eigenvector 3')

plt.show()

In [ ]:
# 2D Projection
fig, ax = plt.subplots(figsize=(8, 6))

sc = ax.scatter(embedding[:, 0], embedding[:, 1], c=angles, cmap=cmap, norm=norm, edgecolor='k', s=50)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Rotation Angle (Degrees)')

ax.set_title('Laplacian Eigenmaps Projection (2D) - Object ' + str(object_label) +' (72 Images)')
ax.set_xlabel('Eigenvector 1')
ax.set_ylabel('Eigenvector 2')

## Weighted Matrix


In [ ]:
from scipy.spatial.distance import cdist


# Euclidean distance matrix
pairwise_distances = cdist(X_subset, X_subset, metric='euclidean')

# affinity matrix using a heat kernel 
sigma = np.mean(pairwise_distances)  # we set sigma as the mean distance
W = np.exp(-pairwise_distances**2 / (2 * sigma**2))

# laplacian
D = np.diag(W.sum(axis=1))  
L = D - W  # unnormalized Laplacian

#  eigenvectors
n_components = 3  # dimensions for embedding
eigvals, eigvecs = eigsh(L, k=n_components+1, which='SM')  # compute smallest eigenvalues and eigenvectors
embedding = eigvecs[:, 1:]  # skip the first eigenvector (it's constant)


fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')


norm = mcolors.Normalize(vmin=0, vmax=360)
cmap = plt.get_cmap('jet')  

sc = ax.scatter(embedding[:, 0], embedding[:, 1], embedding[:, 2], 
                c=angles, cmap=cmap, norm=norm, edgecolor='k', s=50)

cbar = plt.colorbar(sc, ax=ax, shrink=0.6)
cbar.set_label('Rotation Angle (Degrees)')

ax.set_title('Laplacian Eigenmaps Projection (3D)-  object ' + str(object_label) +' (72 Images)')
ax.set_xlabel('Eigenvector 1')
ax.set_ylabel('Eigenvector 2')
ax.set_zlabel('Eigenvector 3')

plt.show()

In [ ]:
# 2D Projection
fig, ax = plt.subplots(figsize=(8, 6))

sc = ax.scatter(embedding[:, 0], embedding[:, 1], c=angles, cmap=cmap, norm=norm, edgecolor='k', s=50)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Rotation Angle (Degrees)')

ax.set_title('Laplacian Eigenmaps Projection (2D) - Object ' + str(object_label) +' (72 Images)')
ax.set_xlabel('Eigenvector 1')
ax.set_ylabel('Eigenvector 2')

plt.show()

## Weighted Matrix with $\varepsilon$ treshold


In [ ]:
 
pairwise_distances = cdist(X_subset, X_subset, metric='euclidean')


 
epsilon =  np.mean(pairwise_distances)  # we set epsilon as the mean distance
print("epsiolon", epsilon)
W = np.exp(-pairwise_distances**2 / (2 * epsilon**2))
W[pairwise_distances > epsilon] = 0   


D = np.diag(W.sum(axis=1))  
L = D - W   
 
n_components = 3   
eigvals, eigvecs = eigsh(L, k=n_components+1, which='SM')   
embedding = eigvecs[:, 1:]  

 
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

norm = mcolors.Normalize(vmin=0, vmax=360)
cmap = plt.get_cmap('jet')   

sc = ax.scatter(embedding[:, 0], embedding[:, 1], embedding[:, 2], 
                c=angles, cmap=cmap, norm=norm, edgecolor='k', s=50)

cbar = plt.colorbar(sc, ax=ax, shrink=0.6)
cbar.set_label('Rotation Angle (Degrees)')

ax.set_title('Laplacian Eigenmaps Projection (3D) - Object ' + str(object_label) +' (72 Images)')
ax.set_xlabel('Eigenvector 1')
ax.set_ylabel('Eigenvector 2')
ax.set_zlabel('Eigenvector 3')

plt.show()

In [ ]:
# 2D Projection
fig, ax = plt.subplots(figsize=(8, 6))

sc = ax.scatter(embedding[:, 0], embedding[:, 1], c=angles, cmap=cmap, norm=norm, edgecolor='k', s=50)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Rotation Angle (Degrees)')

ax.set_title('Laplacian Eigenmaps Projection (2D) -  Object ' + str(object_label) +' (72 Images)')
ax.set_xlabel('Eigenvector 1')
ax.set_ylabel('Eigenvector 2')

plt.show()

## Non-weighted Matrix with $\varepsilon$ treshold


In [ ]:
pairwise_distances = cdist(X_subset, X_subset, metric='euclidean')

epsilon = np.mean(pairwise_distances)
W = (pairwise_distances < epsilon).astype(float)

D = np.diag(W.sum(axis=1))  
L = D - W

n_components = 3
eigvals, eigvecs = eigsh(L, k=n_components+1, which='SM')
embedding = eigvecs[:, 1:]

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

norm = mcolors.Normalize(vmin=0, vmax=360)
cmap = plt.get_cmap('jet')

sc = ax.scatter(embedding[:, 0], embedding[:, 1], embedding[:, 2], 
                c=angles, cmap=cmap, norm=norm, edgecolor='k', s=50)

cbar = plt.colorbar(sc, ax=ax, shrink=0.6)
cbar.set_label('Rotation Angle (Degrees)')

ax.set_title('Laplacian Eigenmaps Projection (3D) - Object ' + str(object_label) +' (72 Images)')
ax.set_xlabel('Eigenvector 1')
ax.set_ylabel('Eigenvector 2')
ax.set_zlabel('Eigenvector 3')

plt.show()

In [ ]:
# 2D Projection
fig, ax = plt.subplots(figsize=(8, 6))

sc = ax.scatter(embedding[:, 0], embedding[:, 1], c=angles, cmap=cmap, norm=norm, edgecolor='k', s=50)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Rotation Angle (Degrees)')

ax.set_title('Laplacian Eigenmaps Projection (2D) -  Object ' + str(object_label) +' (72 Images)')
ax.set_xlabel('Eigenvector 1')
ax.set_ylabel('Eigenvector 2')

plt.show()